# Qwen3-VL-2B QLoRA — 日本語手書き答案採点 (Phase 1')

Kaggle 無料枠の T4 向けに、Task G 形式のデータを Unsloth で QLoRA 学習し、同一の test 集合で LoRA 無効／有効を比較します。実行手順は同じディレクトリの `README.md` を参照してください。


In [ ]:
# 1. 設定: 実験条件はこのセルだけを変更する
from pathlib import Path

DATASET_DIR = Path("/kaggle/input/ja-handwriting-grading-sft")
TRAIN_LIMIT = 3000  # None で train.jsonl の全件
EPOCHS = 1
LORA_R = 16
LR = 2e-4
BATCH = 1
GRAD_ACCUM = 8
MAX_SEQ = 4096
EVAL_N = 200
MAX_PIXELS = 768 * 1086  # T4 向け: 画像トークン数を抑える

MODEL_NAME = "unsloth/Qwen3-VL-2B-Instruct"
ADAPTER_DIR = Path("/kaggle/working/lora_adapters")
RESULTS_PATH = Path("/kaggle/working/eval_results.json")
SEED = 3407
MAX_NEW_TOKENS = 1200
INFERENCE_TEMPERATURE = 0.0

if TRAIN_LIMIT is not None and TRAIN_LIMIT < 0:
    raise ValueError("TRAIN_LIMIT は None または 0 以上にしてください")
if EVAL_N < 0:
    raise ValueError("EVAL_N は 0 以上にしてください")


In [ ]:
# 2. インストール
# Kaggle のベースイメージ更新で依存関係が変わる場合があります。
# エラー時は Internet を有効にしてセッションを再起動し、このセルから再実行してください。
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade", "unsloth",
])


In [ ]:
# 3. 4-bit モデルと LoRA の準備
import torch
from unsloth import FastVisionModel, is_bf16_supported

model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)

# Qwen3-VL processor の画像トークン上限を T4 向けに制限する。
# min_pixels は既定値を尊重しつつ、max_pixels を超えないようにする。
image_processor = getattr(processor, "image_processor", processor)
existing_min_pixels = getattr(image_processor, "min_pixels", 28 * 28 * 64)
image_processor.min_pixels = min(int(existing_min_pixels), MAX_PIXELS)
image_processor.max_pixels = MAX_PIXELS
# processor 自身が同名属性を参照する版にも反映する。
if hasattr(processor, "min_pixels"):
    processor.min_pixels = image_processor.min_pixels
if hasattr(processor, "max_pixels"):
    processor.max_pixels = MAX_PIXELS
print({
    "min_pixels": image_processor.min_pixels,
    "max_pixels": image_processor.max_pixels,
})

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,       # vision tower は凍結
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()


In [ ]:
# 4. データ: JSON だけを保持し、画像は collator 内でバッチごとに遅延ロードする
import copy
import json
import os
from contextlib import ExitStack

from PIL import Image
from torch.utils.data import Dataset
from unsloth.trainer import UnslothVisionDataCollator


def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise TypeError(f"{path}:{line_number}: JSON object ではありません")
            records.append(value)
    return records


def dataset_image_path(record):
    relative = record.get("image")
    if not isinstance(relative, str) or not relative:
        raise ValueError(f"{record.get('id')}: image が不正です")
    root = DATASET_DIR.resolve()
    path = (root / relative).resolve()
    if os.path.commonpath([str(root), str(path)]) != str(root):
        raise ValueError(f"Dataset 外の画像は参照できません: {relative}")
    return path


def conversation_with_image(record, image, include_assistant=True):
    messages = copy.deepcopy(record["messages"])
    if not include_assistant:
        messages = [message for message in messages
                    if message.get("role") != "assistant"]
    replacements = 0
    for message in messages:
        content = message.get("content", [])
        if not isinstance(content, list):
            continue
        for part in content:
            if isinstance(part, dict) and part.get("type") == "image":
                part["image"] = image
                replacements += 1
    if replacements != 1:
        raise ValueError(
            f"{record.get('id')}: image placeholder は 1 個必要です (実際 {replacements})")
    return messages


class LazyJSONLVisionDataset(Dataset):
    """レコードのみ保持する。PIL Image は __getitem__ では開かない。"""

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return {"record": self.records[index]}


class LazyImageVisionCollator:
    """現在のバッチだけ画像を開き、Unsloth collator 完了後に必ず閉じる。"""

    def __init__(self, model, processor):
        self.inner = UnslothVisionDataCollator(model, processor)

    def __call__(self, features):
        with ExitStack() as stack:
            examples = []
            for feature in features:
                record = feature["record"]
                source = stack.enter_context(Image.open(dataset_image_path(record)))
                image = source.convert("RGB")
                stack.callback(image.close)
                examples.append({
                    "messages": conversation_with_image(record, image),
                })
            return self.inner(examples)


train_records = read_jsonl(DATASET_DIR / "train.jsonl")
# 再現性のため shuffle 前の先頭 N 件。None の場合だけ全件。
if TRAIN_LIMIT is not None:
    train_records = train_records[:TRAIN_LIMIT]
if not train_records:
    raise ValueError("学習レコードが 0 件です")

# バンドル契約を早期確認する（val は Trainer では使わないが存在必須）。
for required in ("val.jsonl", "test.jsonl"):
    if not (DATASET_DIR / required).is_file():
        raise FileNotFoundError(DATASET_DIR / required)
for record in train_records:
    path = dataset_image_path(record)
    if not path.is_file():
        raise FileNotFoundError(path)

train_dataset = LazyJSONLVisionDataset(train_records)
vision_collator = LazyImageVisionCollator(model, processor)
print(f"train records: {len(train_dataset)} (images are lazy-loaded per batch)")


In [ ]:
# 5. SFT 学習と adapter 保存
from trl import SFTConfig, SFTTrainer

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=vision_collator,
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_ratio=0.03,
        fp16=not is_bf16_supported(),  # T4
        bf16=is_bf16_supported(),
        gradient_checkpointing=True,
        logging_strategy="steps",
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="/kaggle/working/trainer_outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ,
    ),
)
train_result = trainer.train()  # loss は logging_steps ごとに表示される
print(train_result.metrics)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"saved: {ADAPTER_DIR}")


In [ ]:
# 6. ビフォー／アフター評価
# 指標実装は pipeline/rewards.py および pipeline/run_zeroshot_eval.py と同期。
# この SFT データでは bbox が既に 0-1000 相対座標なので、相対座標同士で比較する。
import gc
import json
import math
import re
import statistics
import unicodedata
from contextlib import nullcontext

import torch
from PIL import Image


def normalize_text(value):
    value = unicodedata.normalize("NFKC", value)
    return re.sub(r"\s+", "", value)


def lev(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + cost)
        prev = cur
    return prev[m]


def cer(ref, hyp):
    return (0.0 if not hyp else 1.0) if not ref else lev(ref, hyp) / len(ref)


def iou(a, b):
    if not a or not b:
        return 0.0
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
    inter = iw * ih
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    denom = area_a + area_b - inter
    return inter / denom if denom > 0 else 0.0


def _mean(values):
    return sum(values) / len(values) if values else None


def parse_json_robust(content):
    if isinstance(content, dict):
        return content
    if not isinstance(content, str):
        raise TypeError("model output is not text or object")
    text = content.strip()
    attempts = [text]
    if text.startswith("```") and text.endswith("```"):
        inner = text[3:-3].strip()
        if inner.lower().startswith("json"):
            inner = inner[4:].lstrip()
        attempts.append(inner)
    for candidate in attempts:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    start = text.find("{")
    if start >= 0:
        value, _ = json.JSONDecoder().raw_decode(text[start:])
        return value
    raise json.JSONDecodeError("JSON object not found", text, 0)


def sanitize_bbox(value):
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        return None
    if any(isinstance(v, bool) or not isinstance(v, (int, float))
           for v in value):
        return None
    coords = [float(v) for v in value]
    if not all(math.isfinite(v) for v in coords):
        return None
    coords = [min(1000.0, max(0.0, v)) for v in coords]
    if coords[2] <= coords[0] or coords[3] <= coords[1]:
        return None
    return [int(v) if v.is_integer() else v for v in coords]


def _as_int(value):
    if isinstance(value, bool) or value is None:
        return None
    try:
        return int(value)
    except (TypeError, ValueError, OverflowError):
        return None


def sanitize_output(value):
    if not isinstance(value, dict):
        raise TypeError("output JSON is not an object")
    if not isinstance(value.get("transcript"), str):
        raise TypeError("transcript is not a string")
    if not isinstance(value.get("errors"), list):
        raise TypeError("errors is not an array")
    errors = []
    for error in value["errors"]:
        if not isinstance(error, dict):
            error = {}
        step_id = error.get("step_id")
        if step_id is not None and not isinstance(step_id, str):
            step_id = str(step_id)
        error_type = error.get("type", "")
        if not isinstance(error_type, str):
            error_type = str(error_type)
        errors.append({
            "step_id": step_id,
            "bbox": sanitize_bbox(error.get("bbox")),
            "type": error_type,
        })
    return {
        "transcript": value["transcript"],
        "errors": errors,
        "score": _as_int(value.get("score")),
    }


def _content_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = [part.get("text", "") for part in content
                 if isinstance(part, dict) and part.get("type") == "text"]
        return "".join(parts)
    raise TypeError("message content is not text")


def teacher_output(record):
    assistants = [message for message in record.get("messages", [])
                  if message.get("role") == "assistant"]
    if len(assistants) != 1:
        raise ValueError(f"{record.get('id')}: assistant message は 1 個必要です")
    return sanitize_output(parse_json_robust(_content_text(assistants[0]["content"])))


def individual_metrics(reference, output):
    has_error = bool(reference["errors"])
    predictions = output["errors"]
    detected = bool(predictions)
    bbox_iou = None
    bbox_hit = None
    if has_error and detected:
        gt_boxes = [error.get("bbox") for error in reference["errors"]
                    if error.get("bbox")]
        bbox_iou = max(
            (iou(prediction.get("bbox"), gt_box)
             for prediction in predictions for gt_box in gt_boxes),
            default=0.0,
        )
        bbox_hit = bbox_iou >= 0.5
    gt_types = [error.get("type", "") for error in reference["errors"]]
    predicted_types = {error.get("type", "") for error in predictions}
    type_hits = [{"type": error_type,
                  "exact_match": error_type in predicted_types}
                 for error_type in gt_types]
    gt_score = reference["score"]
    score = output["score"]
    return {
        "has_error": has_error,
        "detected_error": detected,
        "transcript_cer": cer(normalize_text(reference["transcript"]),
                              normalize_text(output["transcript"])),
        "bbox_iou": bbox_iou,
        "bbox_iou_at_0_5": bbox_hit,
        "type_exact_matches": type_hits,
        "score_exact_match": score == gt_score,
        "score_within_1": (score is not None and gt_score is not None
                           and abs(score - gt_score) <= 1),
    }


def aggregate_metrics(rows):
    parsed = [row for row in rows if not row["parse_failure"]]
    metrics = [row["metrics"] for row in parsed]
    error_metrics = [value for value in metrics if value["has_error"]]
    controls = [value for value in metrics if not value["has_error"]]
    tpr = _mean([float(value["detected_error"]) for value in error_metrics])
    tnr = _mean([float(not value["detected_error"]) for value in controls])
    bacc = ((tpr + tnr) / 2.0
             if tpr is not None and tnr is not None else None)
    type_groups = {}
    for value in metrics:
        for hit in value["type_exact_matches"]:
            type_groups.setdefault(hit["type"], []).append(
                float(hit["exact_match"]))
    type_by_class = {key: _mean(values)
                     for key, values in sorted(type_groups.items())}
    located = [value for value in error_metrics if value["detected_error"]]
    return {
        "n_attempted": len(rows),
        "n_evaluated": len(parsed),
        "parse_failures": len(rows) - len(parsed),
        "transcript_cer_mean": _mean(
            [value["transcript_cer"] for value in metrics]),
        "tpr": tpr,
        "tnr": tnr,
        "bacc": bacc,
        "hallucinated_error_rate": None if tnr is None else 1.0 - tnr,
        "iou_at_0_5": _mean([
            float(value["bbox_iou_at_0_5"]) for value in located
        ]),
        "type_macro_exact_match": _mean(list(type_by_class.values())),
        "type_exact_match_by_class": type_by_class,
        "score_exact_match": _mean([
            float(value["score_exact_match"]) for value in metrics
        ]),
        "score_within_1": _mean([
            float(value["score_within_1"]) for value in metrics
        ]),
        "n_error_records": len(error_metrics),
        "n_control_records": len(controls),
        "n_location_records": len(located),
    }


def _meta_eval_selected(record):
    """Task G の meta に明示された評価選択フラグがあれば拾う。"""
    meta = record.get("meta")
    if not isinstance(meta, dict):
        return False
    for key in ("eval_selected", "selected_for_eval", "is_eval"):
        if meta.get(key) is True:
            return True
    evaluation = meta.get("evaluation")
    if isinstance(evaluation, dict) and evaluation.get("selected") is True:
        return True
    return meta.get("split") in {"eval", "evaluation"}


def select_eval_records(records, count):
    # meta 由来の選択分を入力順で優先し、不足分を test.jsonl の先頭から補う。
    priority = [record for record in records if _meta_eval_selected(record)]
    chosen = []
    seen = set()
    for record in priority + records:
        key = record.get("id")
        if key in seen:
            continue
        chosen.append(record)
        seen.add(key)
        if len(chosen) >= count:
            break
    return chosen


def infer_one(record):
    image_path = dataset_image_path(record)
    with Image.open(image_path) as source:
        image = source.convert("RGB")
        try:
            messages = conversation_with_image(
                record, image, include_assistant=False)
            prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(
                text=[prompt], images=[image], return_tensors="pt")
        finally:
            image.close()
    device = next(model.parameters()).device
    inputs = {key: value.to(device) if hasattr(value, "to") else value
              for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[1]
    # do_sample=False は generation API における temperature=0 の決定的推論。
    generation_kwargs = {
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
        "use_cache": True,
    }
    with torch.inference_mode():
        generated = model.generate(**inputs, **generation_kwargs)
    return processor.batch_decode(
        generated[:, input_length:], skip_special_tokens=True)[0]


def evaluate_records(records, adapter_enabled):
    if not hasattr(model, "disable_adapter"):
        raise RuntimeError(
            "この PEFT 版には disable_adapter() がありません。依存関係を更新してください")
    adapter_context = (nullcontext() if adapter_enabled
                       else model.disable_adapter())
    rows = []
    with adapter_context:
        for index, record in enumerate(records, 1):
            raw = ""
            parsed_output = None
            failure_reason = None
            reference = teacher_output(record)
            try:
                raw = infer_one(record)
                parsed_output = sanitize_output(parse_json_robust(raw))
            except Exception as exc:
                failure_reason = f"{type(exc).__name__}: {exc}"
            rows.append({
                "id": record.get("id"),
                "parse_failure": failure_reason is not None,
                "failure_reason": failure_reason,
                "raw_output": raw,
                "parsed_output": parsed_output,
                "metrics": (individual_metrics(reference, parsed_output)
                            if parsed_output is not None else None),
            })
            if index % 10 == 0 or index == len(records):
                failures = sum(row["parse_failure"] for row in rows)
                print(f"  {index}/{len(records)} parse_failures={failures}",
                      flush=True)
    summary = aggregate_metrics(rows)
    summary["temperature"] = INFERENCE_TEMPERATURE
    summary["max_new_tokens"] = MAX_NEW_TOKENS
    return {"summary": summary, "rows": rows}


test_records = read_jsonl(DATASET_DIR / "test.jsonl")
eval_records = select_eval_records(test_records, EVAL_N)
if not eval_records:
    raise ValueError("評価レコードが 0 件です")
print(f"eval records: {len(eval_records)} / test records: {len(test_records)}")

FastVisionModel.for_inference(model)
print("[before] base model (LoRA disabled)")
before_result = evaluate_records(eval_records, adapter_enabled=False)
gc.collect()
torch.cuda.empty_cache()
print("[after] trained LoRA enabled")
after_result = evaluate_records(eval_records, adapter_enabled=True)


In [ ]:
# 7. 結果表示と保存
# EVO-X2 2B zero-shot 参考値:
# CER=0.024, BACC=0.843, 幻覚率=0.314, 点数完全一致=0.345,
# IoU@0.5=0.0, 種別完全一致=0.0

metric_rows = [
    ("transcript CER (mean)", "transcript_cer_mean"),
    ("TPR", "tpr"),
    ("TNR", "tnr"),
    ("BACC", "bacc"),
    ("hallucination rate", "hallucinated_error_rate"),
    ("score exact", "score_exact_match"),
    ("score ±1", "score_within_1"),
    ("bbox IoU@0.5", "iou_at_0_5"),
    ("type exact (macro)", "type_macro_exact_match"),
    ("parse failures", "parse_failures"),
]


def display_value(value):
    if value is None:
        return "N/A"
    if isinstance(value, float):
        return f"{value:.6f}"
    return str(value)


before_summary = before_result["summary"]
after_summary = after_result["summary"]
print(f"{'metric':<28} {'before':>12} {'after':>12}")
print("-" * 54)
for label, key in metric_rows:
    print(f"{label:<28} {display_value(before_summary.get(key)):>12} "
          f"{display_value(after_summary.get(key)):>12}")

results = {
    "model": MODEL_NAME,
    "adapter_dir": str(ADAPTER_DIR),
    "config": {
        "train_limit": TRAIN_LIMIT,
        "epochs": EPOCHS,
        "lora_r": LORA_R,
        "learning_rate": LR,
        "batch": BATCH,
        "gradient_accumulation": GRAD_ACCUM,
        "max_seq": MAX_SEQ,
        "eval_n": EVAL_N,
        "max_pixels": MAX_PIXELS,
    },
    "eval_ids": [record.get("id") for record in eval_records],
    "before": before_result,
    "after": after_result,
    "reference_baseline_evo_x2_2b": {
        "transcript_cer_mean": 0.024,
        "bacc": 0.843,
        "hallucinated_error_rate": 0.314,
        "score_exact_match": 0.345,
        "iou_at_0_5": 0.0,
        "type_macro_exact_match": 0.0,
    },
}
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, "w", encoding="utf-8") as stream:
    json.dump(results, stream, ensure_ascii=False, indent=2)
print(f"saved: {RESULTS_PATH}")
